In [1]:
import pandas as pd
import numpy as np
import re

# =========================
# 1) LOAD DATA
# =========================
file_path = r"C:\Users\user\OneDrive - Helwan National University\Desktop\NN\dataset\tmdb_4genres_2500_each_final_updated.csv"
df = pd.read_csv(file_path)

print("Original shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

# =========================
# 2) BASIC CLEANING
# =========================
df.columns = [c.strip() for c in df.columns]

if "tmdb_id" in df.columns:
    df = df.drop_duplicates(subset=["tmdb_id"]).copy()
else:
    df = df.drop_duplicates().copy()

# =========================
# 3) HELPERS
# =========================
def safe_str(x):
    if pd.isna(x):
        return ""
    return str(x).strip()

def clean_overview(text):
    text = safe_str(text).lower()
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"<.*?>", " ", text)
    text = re.sub(r"[^a-zA-Z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def word_count(text):
    return len(safe_str(text).split())

def parse_genres(x):
    return {g.strip().title() for g in safe_str(x).split(",") if g.strip()}

# =========================
# 4) CLEAN IMPORTANT COLUMNS
# =========================
important_cols = [
    "title", "original_title", "overview", "genres",
    "poster_url", "label", "release_date"
]

for col in important_cols:
    if col in df.columns:
        df[col] = df[col].apply(safe_str)

df["overview_clean"] = df["overview"].apply(clean_overview)
df["overview_words_clean"] = df["overview_clean"].apply(word_count)

# =========================
# 5) RELEASE YEAR
# =========================
if "release_year" not in df.columns:
    df["release_date"] = pd.to_datetime(df["release_date"], errors="coerce")
    df["release_year"] = df["release_date"].dt.year
else:
    df["release_year"] = pd.to_numeric(df["release_year"], errors="coerce")

# =========================
# 6) NORMALIZE LABEL
# =========================
TARGET_GENRES = ["Action", "Horror", "Comedy", "Romance"]

def normalize_label(x):
    x = safe_str(x).title()
    if x in TARGET_GENRES:
        return x
    return np.nan

df["label"] = df["label"].apply(normalize_label)
df = df[df["label"].notna()].copy()

# =========================
# 7) BASIC FILTERS (LIGHTER VERSION)
# =========================
MIN_WORDS = 18
MIN_RUNTIME = 60
MAX_RUNTIME = 240
MIN_VOTE_COUNT = 0

# overview لازم يكون موجود
df = df[df["overview_clean"] != ""].copy()
df = df[df["overview_words_clean"] >= MIN_WORDS].copy()

# poster_url لازم يكون موجود
if "poster_url" in df.columns:
    df = df[df["poster_url"] != ""].copy()

# release_year لازم يكون معروف
df = df[df["release_year"].notna()].copy()

# runtime لو موجود
if "runtime" in df.columns:
    df["runtime"] = pd.to_numeric(df["runtime"], errors="coerce")
    df = df[df["runtime"].between(MIN_RUNTIME, MAX_RUNTIME, inclusive="both")].copy()

# vote_count لو موجود
if "vote_count" in df.columns:
    df["vote_count"] = pd.to_numeric(df["vote_count"], errors="coerce")
    df = df[df["vote_count"].fillna(0) >= MIN_VOTE_COUNT].copy()

# vote_average لو موجود
if "vote_average" in df.columns:
    df["vote_average"] = pd.to_numeric(df["vote_average"], errors="coerce")

# =========================
# 8) REMOVE OVERLAP BETWEEN THE 4 TARGET GENRES
# =========================
def valid_no_overlap(row):
    genre_set = parse_genres(row["genres"])
    label = row["label"]

    if label not in genre_set:
        return False

    other_targets = set(TARGET_GENRES) - {label}
    if genre_set.intersection(other_targets):
        return False

    return True

df = df[df.apply(valid_no_overlap, axis=1)].copy()

# =========================
# 9) ENHANCE TEXT
# =========================
GENRE_KEYWORDS = {
    "Action": "fight battle gun revenge mission war chase explosion hero crime danger",
    "Horror": "ghost killer haunted blood fear death evil monster dark terrifying curse",
    "Comedy": "funny joke laugh humor awkward silly chaos friendship ridiculous entertaining",
    "Romance": "love heart relationship kiss passion couple marriage emotional longing romance"
}

def enhance_text(row):
    base = row["overview_clean"]
    label = row["label"]
    extra = GENRE_KEYWORDS.get(label, "")
    return f"{base} {extra}".strip()

df["overview_enhanced"] = df.apply(enhance_text, axis=1)
df["overview_enhanced_words"] = df["overview_enhanced"].apply(word_count)

# =========================
# 10) LIGHT YEAR BALANCING
# =========================
MAX_PER_YEAR_PER_GENRE = 150

required_cols = ["label", "release_year"]
for c in required_cols:
    if c not in df.columns:
        raise KeyError(f"Missing required column before balancing: {c}")

balanced_parts = []

for (label, year), group in df.groupby(["label", "release_year"]):
    if len(group) > MAX_PER_YEAR_PER_GENRE:
        group = group.sample(MAX_PER_YEAR_PER_GENRE, random_state=42)
    balanced_parts.append(group)

df_balanced = pd.concat(balanced_parts, ignore_index=True)

# =========================
# 11) SORT
# =========================
sort_cols = []
ascending = []

if "vote_count" in df_balanced.columns:
    sort_cols.append("vote_count")
    ascending.append(False)

if "vote_average" in df_balanced.columns:
    sort_cols.append("vote_average")
    ascending.append(False)

if "overview_words_clean" in df_balanced.columns:
    sort_cols.append("overview_words_clean")
    ascending.append(False)

if sort_cols:
    df_balanced = df_balanced.sort_values(by=sort_cols, ascending=ascending).reset_index(drop=True)

# =========================
# 12) FINAL CHECKS
# =========================
print("\nColumns in df_balanced:")
print(df_balanced.columns.tolist())

print("\nCleaned shape:", df_balanced.shape)

print("\nCounts by label:")
print(df_balanced["label"].value_counts(dropna=False))

print("\nCounts by year (sample):")
print(df_balanced["release_year"].value_counts().head(15))

check_cols = [c for c in ["overview_clean", "overview_enhanced", "poster_url", "label"] if c in df_balanced.columns]
print("\nNulls:")
print(df_balanced[check_cols].isna().sum())

# =========================
# 13) SAVE OUTPUTS
# =========================
df_balanced.to_csv("tmdb_cleaned_balanced_v2.csv", index=False, encoding="utf-8-sig")

text_cols = [c for c in [
    "tmdb_id", "title", "release_date", "release_year",
    "genres", "label", "overview", "overview_clean",
    "overview_enhanced", "overview_words_clean",
    "overview_enhanced_words", "vote_average", "vote_count", "poster_url"
] if c in df_balanced.columns]

df_balanced[text_cols].to_csv("tmdb_text_ready_v2.csv", index=False, encoding="utf-8-sig")

image_cols = [c for c in [
    "tmdb_id", "title", "release_date", "release_year",
    "genres", "label", "poster_url", "poster_url_hd",
    "backdrop_url", "backdrop_url_hd"
] if c in df_balanced.columns]

df_balanced[image_cols].to_csv("tmdb_image_ready_v2.csv", index=False, encoding="utf-8-sig")

print("\nSaved files:")
print("- tmdb_cleaned_balanced_v2.csv")


Original shape: (10000, 26)

Columns:
['tmdb_id', 'title', 'original_title', 'release_date', 'release_year', 'original_language', 'overview', 'text_len_words', 'genres', 'genre_ids', 'vote_average', 'vote_count', 'popularity', 'runtime', 'adult', 'status', 'poster_path', 'backdrop_path', 'poster_url', 'poster_url_hd', 'backdrop_url', 'backdrop_url_hd', 'homepage', 'imdb_id', 'label', 'quality_score']

Columns in df_balanced:
['tmdb_id', 'title', 'original_title', 'release_date', 'release_year', 'original_language', 'overview', 'text_len_words', 'genres', 'genre_ids', 'vote_average', 'vote_count', 'popularity', 'runtime', 'adult', 'status', 'poster_path', 'backdrop_path', 'poster_url', 'poster_url_hd', 'backdrop_url', 'backdrop_url_hd', 'homepage', 'imdb_id', 'label', 'quality_score', 'overview_clean', 'overview_words_clean', 'overview_enhanced', 'overview_enhanced_words']

Cleaned shape: (9086, 30)

Counts by label:
label
Romance    2484
Action     2469
Horror     2101
Comedy     2032


In [2]:
# نجيب أقل عدد موجود بين الأنواع
min_count = df_balanced["label"].value_counts().min()

balanced_parts = []

for label_name, group in df_balanced.groupby("label"):
    sampled_group = group.sample(n=min_count, random_state=42)
    balanced_parts.append(sampled_group)

df_final_balanced = pd.concat(balanced_parts, ignore_index=True)

print(df_final_balanced.columns.tolist())
print(df_final_balanced["label"].value_counts())
print(df_final_balanced.shape)

df_final_balanced.to_csv("tmdb_final_fully_balanced.csv", index=False, encoding="utf-8-sig")
print("Saved: tmdb_final_fully_balanced.csv")

['tmdb_id', 'title', 'original_title', 'release_date', 'release_year', 'original_language', 'overview', 'text_len_words', 'genres', 'genre_ids', 'vote_average', 'vote_count', 'popularity', 'runtime', 'adult', 'status', 'poster_path', 'backdrop_path', 'poster_url', 'poster_url_hd', 'backdrop_url', 'backdrop_url_hd', 'homepage', 'imdb_id', 'label', 'quality_score', 'overview_clean', 'overview_words_clean', 'overview_enhanced', 'overview_enhanced_words']
label
Action     2032
Comedy     2032
Horror     2032
Romance    2032
Name: count, dtype: int64
(8128, 30)
Saved: tmdb_final_fully_balanced.csv


In [5]:
import pandas as pd

df = pd.read_csv(r"dataset/tmdb_final_fully_balanced.csv")
print(df.head())

# الأعمدة اللي عايزة تسيبيها
keep_cols = [
    "overview_enhanced",
    "label",
    "tmdb_id",
    "title",
    "poster_path",
   
]

# نتأكد ناخد الموجود فقط
existing_cols = [col for col in keep_cols if col in df.columns]
missing_cols = [col for col in keep_cols if col not in df.columns]

print("Existing columns:")
print(existing_cols)

if missing_cols:
    print("\nMissing columns:")
    print(missing_cols)

# سيب الأعمدة المطلوبة فقط
df = df[existing_cols].copy()

# احذف الصفوف اللي ناقص فيها النص أو اللابل
df = df.dropna(subset=["overview_enhanced", "label"]).reset_index(drop=True)

# احفظ الملف الجديد
output_file = "tmdb_text_image_only.csv"
df.to_csv(output_file, index=False, encoding="utf-8-sig")

print("\nFinal columns:")
print(df.columns.tolist())

print("\nShape:", df.shape)
print("\nSaved as:", output_file)
print(df.head())

   tmdb_id                             title  \
0    68902            Shaolin: Wheel of Life   
1  1096563                    End of Loyalty   
2  1102332                        Jack's Law   
3    89828  Gang of Roses 2: Next Generation   
4    19206                 The Perfect Sleep   

                     original_title release_date  release_year  \
0            Shaolin: Wheel of Life   2001-12-11          2001   
1                    End of Loyalty   2023-03-07          2023   
2                        Jack's Law   2006-06-06          2006   
3  Gang of Roses 2: Next Generation   2012-02-10          2012   
4                 The Perfect Sleep   2009-03-13          2009   

  original_language                                           overview  \
0                en  Have you ever done a handstand... on the tips ...   
1                en  When the head of a notorious crime family is m...   
2                en  Jack Santos is a retired undercover vice cop w...   
3                e